In [1]:
import ee
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from pathlib import Path


# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project="acm-research-f26")
print("Google Earth Engine initialized successfully.")
import numpy as np
import boto3

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)


Google Earth Engine initialized successfully.


In [3]:
#load facility metadata directly from S3.
BUCKET = "atlas-acm-research-f26"
SOURCE_METADATA_KEY = "dataset/im3_atlas_cleaned.csv"

session = boto3.Session(profile_name="ATLAS")
s3 = session.client("s3", region_name="us-east-2")

csv_object = s3.get_object(
    Bucket=BUCKET,
    Key=SOURCE_METADATA_KEY,
)

try:
    gdf = pd.read_csv(
        BytesIO(csv_object["Body"].read()),
        dtype={"id": str},
    )
finally:
    csv_object["Body"].close()

#keep facilities supported by the imagery pipeline.
gdf = gdf[gdf["type"].isin(["building", "campus", "point"])].copy()

#prepare the coordinates used to retrieve NAIP imagery.
gdf["id"] = gdf["id"].str.strip()
gdf["lat"] = pd.to_numeric(gdf["lat"], errors="coerce")
gdf["lon"] = pd.to_numeric(gdf["lon"], errors="coerce")
gdf = gdf.dropna(subset=["lat", "lon"])

print(f"Loaded metadata from s3://{BUCKET}/{SOURCE_METADATA_KEY}")
print("Total usable records:", len(gdf))
print(gdf["type"].value_counts())

Loaded metadata from s3://atlas-acm-research-f26/dataset/im3_atlas_cleaned.csv
Total usable records: 317
type
building    247
campus       70
Name: count, dtype: int64


In [4]:
def get_latest_naip_image(region, start_year=2026, earliest_year=2020):
    for year in range(start_year, earliest_year - 1, -1):
        naip = (
            ee.ImageCollection("USDA/NAIP/DOQQ")
            .filterBounds(region)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .sort("system:time_start", False)

        )
        image_count = naip.size().getInfo()
        print(f"Year: {year}, Image Count: {image_count}")

        if image_count > 0:
            newest = ee.Image(naip.first())
            date = (
                ee.Date(newest.get("system:time_start"))
                .format("YYYY-MM-dd")
                .getInfo()
            )
            image = naip.mosaic()
            return image, date, year
    return None, None, None

In [5]:
#checkpoint download
from huggingface_hub import hf_hub_download

model_name = "ViT-B-32"

checkpoint_path = hf_hub_download(
    repo_id="chendelong/RemoteCLIP",
    filename=f"RemoteCLIP-{model_name}.pt",
    cache_dir=PROJECT_ROOT / "checkpoints"
)

print("Checkpoint:", checkpoint_path)

Checkpoint: /Users/kevinlam/ATLAS/checkpoints/models--chendelong--RemoteCLIP/snapshots/bf1d8a3ccf2ddbf7c875705e46373bfe542bce38/RemoteCLIP-ViT-B-32.pt


In [6]:
#load RemoteCLIP model
import torch
import open_clip

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using:", device)

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name
)

tokenizer = open_clip.get_tokenizer(model_name)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu"
)

message = model.load_state_dict(ckpt)
print(message)

model = model.to(device).eval()

Using: mps


<All keys matched successfully>


In [7]:
#NAIP --> RemoteCLIP

valid_queries = [
    "a close overhead aerial photograph centered on one completed data center building with its entire roof visible",
    "a high resolution aerial photograph of a completed data center facility with clearly defined building boundaries",
    "an overhead aerial photograph of a large completed industrial building occupying most of the image",
    "a close aerial view of a completed data center campus with large rectangular buildings and visible cooling equipment",
    "an aerial photograph of a completed industrial facility with its roof, surrounding access roads, and infrastructure clearly visible",
]

invalid_queries = [
    "a wide aerial photograph of a dense urban area with highways, office buildings, and parking lots",
    "an aerial photograph of a residential neighborhood containing many small houses and local streets",
    "an overhead aerial photograph of a city block containing many unrelated buildings with no single dominant facility",
    "an aerial photograph of a commercial district with shopping centers, parking lots, and busy roads",
    "an aerial photograph of an industrial area where the target facility is too small to distinguish",
    "an aerial photograph of a construction site containing unfinished buildings, exposed foundations, and bare soil",
    "an overhead aerial photograph of an incomplete industrial building with an unfinished roof",
]



In [9]:


# AWS S3 configuration
session = boto3.Session(profile_name="ATLAS")

s3 = session.client(
    "s3",
    region_name="us-east-2"
)

BUCKET = "atlas-acm-research-f26"
S3_PREFIX = "satellite/raw/"

# Configuration
NUM_IMAGES = 317

temp_folder = PROJECT_ROOT / "Data/temp"
Rtemp_folder = PROJECT_ROOT / "Data/rejectedTemp"
Rtemp_folder.mkdir(parents=True, exist_ok=True)

temp_folder.mkdir(parents=True, exist_ok=True)

#existing ids in S3
existing_ids = set()

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        filename = obj["Key"].removeprefix(S3_PREFIX)

        if filename.endswith(".png") and "/" not in filename:
            existing_ids.add(Path(filename).stem)

# Process facilities sequentially in the same order as the S3 CSV.
data_centers = (
    gdf.loc[~gdf["id"].isin(existing_ids)]
    .head(NUM_IMAGES)
)

print("Already in S3:", len(existing_ids))
print("Images selected:", len(data_centers))

# encode RemoteCLIP text descriptions once
text_queries = valid_queries + invalid_queries
text_input = tokenizer(text_queries).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_input)
    text_features /= text_features.norm(dim=-1, keepdim=True)

results = []

for count, (_, data_center) in enumerate(data_centers.iterrows(), start=1):

    center_id = str(data_center["id"])

    print(f"\n[{count}/{len(data_centers)}] {center_id}")

    result = {
        "id": center_id,
        "date": None,
        "decision": "REJECT",
        "reason": None,
        "margin": None,
        "image_path": None,
        "s3_key": None,
        "upload_status": "NOT_UPLOADED"
    }

    try:
        # get coordinates and buffer size
        lon = float(data_center["lon"])
        lat = float(data_center["lat"])

        buffer_meters = 300

        building_ee = ee.Geometry.Point([lon, lat])
        region = building_ee.buffer(buffer_meters).bounds()

        #retrieve NAIP imagery
        image, date, _ = get_latest_naip_image(region)

        if image is None:
            result["reason"] = "NO_NAIP_IMAGE"
            print("REJECT: No NAIP imagery")
            results.append(result)
            continue

        result["date"] = date

        #same dimensions for all images
        IMAGE_SIZE = 1000  

        rgb = image.select(["R", "G", "B"])

        url = rgb.getThumbURL({
            "bands": ["R", "G", "B"],
            "min": 0,
            "max": 255,
            "dimensions": f"{IMAGE_SIZE}x{IMAGE_SIZE}",
            "region": region,
            "format": "png"
        })

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        satellite_image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")


        if satellite_image.size != (IMAGE_SIZE, IMAGE_SIZE):
            satellite_image = satellite_image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

        #check for blank or invalid imagery
        pixels = np.asarray(satellite_image)

        if (
            pixels.size == 0
            or pixels.std() < 5
            or pixels.mean() < 5
            or pixels.mean() > 250
        ):
            result["reason"] = "INVALID_IMAGE"
            print("REJECT: Invalid image")
            results.append(result)
            continue

        # encode image using RemoteCLIP
        image_input = (
            preprocess(satellite_image)
            .unsqueeze(0)
            .to(device)
        )

        with torch.no_grad():
            image_features = model.encode_image(image_input)

            image_features /= image_features.norm(
                dim=-1,
                keepdim=True
            )

            similarities = (
                image_features @ text_features.T
            )[0]


        num_valid = len(valid_queries)

        best_valid = similarities[:num_valid].max().item()
        best_invalid = similarities[num_valid:].max().item()

        margin = best_valid - best_invalid
        result["margin"] = margin

        decision = "KEEP" if margin > 0 else "REJECT"

        result["decision"] = decision
        result["reason"] = "REMOTECLIP_CLASSIFICATION"

        file_path = temp_folder / f"{center_id}.png"
        Rfile_path = Rtemp_folder / f"{center_id}.png"

        if decision == "KEEP":

            #saved locally for manual review
            satellite_image.save(file_path)
            result["image_path"] = str(file_path)

            # upload accepted image to S3
            s3_key = f"{S3_PREFIX}{center_id}.png"

            try:
                s3.upload_file(
                    str(file_path),
                    BUCKET,
                    s3_key,
                    ExtraArgs={"ContentType": "image/png"}
                )

                result["s3_key"] = s3_key
                result["upload_status"] = "SUCCESS"

                print(f"KEEP | Margin: {margin:.4f}")
                print(f"Uploaded: s3://{BUCKET}/{s3_key}")

            except Exception as upload_error:
                result["upload_status"] = "FAILED"
                result["upload_error"] = str(upload_error)

                print(f"KEEP | S3 upload failed: {upload_error}")

        else:

            if file_path.exists():
                file_path.unlink()

            satellite_image.save(Rfile_path)
            result["image_path"] = str(Rfile_path)
            result["upload_status"] = "NOT_UPLOADED"

            print(f"REJECT | Margin: {margin:.4f}")

        # record every processed image
        results.append(result)

    except Exception as e:
        result["decision"] = "ERROR"
        result["reason"] = str(e)

        results.append(result)

        print(f"ERROR: {e}")

#save validation results
results_df = pd.DataFrame(results)

results_df.to_csv(
    PROJECT_ROOT / "Data/validation_results.csv",
    index=False
)

print("\n--- BATCH SUMMARY ---")
print(results_df["decision"].value_counts())

print(
    "Accepted images:",
    (results_df["decision"] == "KEEP").sum()
)

print(
    "Successfully uploaded:",
    (results_df["upload_status"] == "SUCCESS").sum()
)


Already in S3: 0
Images selected: 317

[1/317] 0007805491
Year: 2026, Image Count: 0
Year: 2025, Image Count: 0
Year: 2024, Image Count: 0
Year: 2023, Image Count: 2
KEEP | Margin: 0.0135
Uploaded: s3://atlas-acm-research-f26/satellite/raw/0007805491.png

[2/317] 0009474864
Year: 2026, Image Count: 0
Year: 2025, Image Count: 0
Year: 2024, Image Count: 0
Year: 2023, Image Count: 0
Year: 2022, Image Count: 1
REJECT | Margin: -0.0196

[3/317] 0013924557
Year: 2026, Image Count: 0
Year: 2025, Image Count: 0
Year: 2024, Image Count: 0
Year: 2023, Image Count: 1
REJECT | Margin: -0.0100

[4/317] 0014593270
Year: 2026, Image Count: 0
Year: 2025, Image Count: 0
Year: 2024, Image Count: 0
Year: 2023, Image Count: 0
Year: 2022, Image Count: 1
REJECT | Margin: -0.0125

[5/317] 0014930068
Year: 2026, Image Count: 0
Year: 2025, Image Count: 0
Year: 2024, Image Count: 0
Year: 2023, Image Count: 0
Year: 2022, Image Count: 1
REJECT | Margin: -0.0152

[6/317] 0014997588
Year: 2026, Image Count: 0
Year:

## NAIP metadata recovery to S3

In [10]:
#configuration for recovery of metadata for existing raw S3 images.
BUCKET = 'atlas-acm-research-f26'
PREFIX = 'satellite/raw/'
SOURCE_METADATA_KEY = 'dataset/im3_atlas_cleaned.csv'
OUTPUT_PREFIX = 'satellite/metadata/'
OUTPUT_CSV = f'{OUTPUT_PREFIX}naip_metadata_recovered.csv'
AWS_PROFILE = 'ATLAS'
EE_PROJECT = 'acm-research-f26'
EARLIEST_YEAR = 2020
LATEST_YEAR = 2026
BUFFER_METERS = 300


In [11]:
#recover NAIP region and scene metadata for existing ATLAS S3 PNGs.

import json
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import struct
import boto3
import ee
import pandas as pd


def normalize_metadata_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return str(int(value)) if value.isdecimal() else value


def recover_naip_metadata():
    ee.Initialize(project=EE_PROJECT)
    session = boto3.Session(profile_name=AWS_PROFILE)
    s3 = session.client('s3')
    keys = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=BUCKET, Prefix=PREFIX):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith('.png') and '/' not in key[len(PREFIX):]:
                keys.append(key)
    csv_object = s3.get_object(Bucket=BUCKET, Key=SOURCE_METADATA_KEY)
    try:
        df = pd.read_csv(BytesIO(csv_object['Body'].read()), dtype={'id': str})
    finally:
        csv_object['Body'].close()
    df['id'] = df['id'].map(normalize_metadata_id)
    df = df[df['type'].isin(['building','campus','point'])].copy()
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
    df = df.dropna(subset=['lat','lon']).drop_duplicates('id').set_index('id')
    records = []
    for index,key in enumerate(keys,1):
        fid = Path(key).stem
        rec = dict(id=fid,s3_key=key,status='OK',region_geojson='',bounds_wgs84='',candidate_scene_ids='',candidate_scene_dates='',scene_count=0,newest_candidate_date='',year='',notes='')
        rec.update(metadata_provenance='reconstructed_not_verified_historical_provenance',
                   source_metadata_key=SOURCE_METADATA_KEY,
                   recovered_at_utc=datetime.now(timezone.utc).isoformat())
        try:
            png_object = s3.get_object(Bucket=BUCKET, Key=key, Range='bytes=0-31')
            try:
                header = png_object['Body'].read()
            finally:
                png_object['Body'].close()
            if header[:8] != b'\x89PNG\r\n\x1a\n' or len(header) < 24:
                raise ValueError('Invalid PNG header')
            width, height = struct.unpack('>II', header[16:24])
            lookup_id = normalize_metadata_id(fid)
            if lookup_id not in df.index:
                raise ValueError('Facility ID absent from source CSV')
            row = df.loc[lookup_id]
            lat,lon = float(row['lat']),float(row['lon'])
            region = ee.Geometry.Point([lon,lat]).buffer(BUFFER_METERS).bounds()
            geom = region.getInfo()
            rec.update(lat=lat,lon=lon,buffer_meters=BUFFER_METERS,region_geojson=json.dumps(geom),bounds_wgs84=json.dumps(geom['coordinates']),image_width_px=width,image_height_px=height)
            for year in range(LATEST_YEAR,EARLIEST_YEAR-1,-1):
                col = (ee.ImageCollection('USDA/NAIP/DOQQ').filterBounds(region)
                       .filterDate(f'{year}-01-01',f'{year+1}-01-01')
                       .sort('system:time_start',False))
                count = col.size().getInfo()
                if count:
                    info = ee.Dictionary({
                        'ids': col.aggregate_array('system:index'),
                        'timestamps': col.aggregate_array('system:time_start')
                    }).getInfo()
                    dates = [
                        pd.to_datetime(timestamp, unit='ms', utc=True).isoformat()
                        if timestamp is not None else None
                        for timestamp in info['timestamps']
                    ]
                    rec['year'] = year
                    rec['scene_count'] = count
                    rec['candidate_scene_ids'] = json.dumps([
                        f'USDA/NAIP/DOQQ/{scene_id}' for scene_id in info['ids']
                    ])
                    rec['candidate_scene_dates'] = json.dumps(dates)
                    rec['newest_candidate_date'] = dates[0] if dates else None
                    rec['notes'] = 'Candidate scenes and crop are reconstructed from current selection logic and CSV; historical provenance is unverified. Mosaic can contain multiple scenes; newest candidate date is not a date for every pixel. This is not a building footprint.'
                    break
            else:
                rec['status']='NO_SCENE_IN_YEAR_RANGE'
        except Exception as exc:
            rec['status']='ERROR'
            rec['notes']=str(exc)
        records.append(rec)
        print(f'{index}/{len(keys)} {fid}: {rec["status"]}',flush=True)
        s3.put_object(Bucket=BUCKET, Key=f'{OUTPUT_PREFIX}{fid}.json',
                      Body=json.dumps(rec, indent=2).encode('utf-8'), ContentType='application/json')
        s3.put_object(Bucket=BUCKET, Key=OUTPUT_CSV,
                      Body=pd.DataFrame(records).to_csv(index=False).encode('utf-8'),
                      ContentType='text/csv')
    print(f'Saved {len(records)} rows to s3://{BUCKET}/{OUTPUT_CSV}')
    return pd.DataFrame(records)

recovered_metadata = recover_naip_metadata()


1/180 0007805491: OK
2/180 0014593270: OK
3/180 0014930068: OK
4/180 0014997588: OK
5/180 0015884451: OK
6/180 0016282459: OK
7/180 0018518989: OK
8/180 0025706343: OK
9/180 0028544618: OK
10/180 0029827954: OK
11/180 0031800424: OK
12/180 0032560290: OK
13/180 0034770750: OK
14/180 0052227492: OK
15/180 0079818498: OK
16/180 0095349317: OK
17/180 0122665255: OK
18/180 0128499684: OK
19/180 0131891022: OK
20/180 0138260136: OK
21/180 0138643160: OK
22/180 0155739337: OK
23/180 0161126131: OK
24/180 0168129695: OK
25/180 0195165354: OK
26/180 0195664350: OK
27/180 0199252950: OK
28/180 0203432103: OK
29/180 0204316962: OK
30/180 0209786637: OK
31/180 0220518160: OK
32/180 0225745958: OK
33/180 0231769626: OK
34/180 0239032142: OK
35/180 0240724127: OK
36/180 0254773884: OK
37/180 0256728809: OK
38/180 0260996040: OK
39/180 0269166415: OK
40/180 0269847438: OK
41/180 0278560778: OK
42/180 0283051472: OK
43/180 0293211686: OK
44/180 0300952513: OK
45/180 0314032203: OK
46/180 0317466755: 

86/180 0598261190: OK
87/180 0609126465: OK
88/180 0610845529: OK
89/180 0612045611: OK
90/180 0626226408: OK
91/180 0633034071: OK
92/180 0664938835: OK
93/180 0667796922: OK
94/180 0675108684: OK
95/180 0680666647: OK
96/180 0696992451: OK
97/180 0697021949: OK
98/180 0700001774: OK
99/180 0701931820: OK
100/180 0706725650: OK
101/180 0708931343: OK
102/180 0713411278: OK
103/180 0714120969: OK
104/180 0714139868: OK
105/180 0714622339: OK
106/180 0714776107: OK
107/180 0714889192: OK
108/180 0714943309: OK
109/180 0715098032: OK
110/180 0718465287: OK
111/180 0729376485: OK
112/180 0733721782: OK
113/180 0741629998: OK
114/180 0744177375: OK
115/180 0746278786: OK
116/180 0773848531: OK
117/180 0777196805: OK
118/180 0795853621: OK
119/180 0815057182: OK
120/180 0819981888: OK
121/180 0830138727: OK
122/180 0830885779: OK
123/180 0836908998: OK
124/180 0836938516: OK
125/180 0840582927: OK
126/180 0844359206: OK
127/180 0844386860: OK
128/180 0845775548: OK
129/180 0880052062: OK
13

159/180 1143804764: OK
160/180 1167564435: OK
161/180 1178609569: OK
162/180 1207052719: OK
163/180 1210421871: OK
164/180 1235150788: OK
165/180 1238705019: OK
166/180 1252196823: OK
167/180 1252196824: OK
168/180 1254161871: OK
169/180 1280344149: OK
170/180 1282332209: OK
171/180 1292169104: OK
172/180 1307072785: OK
173/180 1307920829: OK
174/180 1312792611: OK
175/180 1318322780: OK
176/180 1319724559: OK
177/180 1343100173: OK
178/180 1344366214: OK
179/180 1346610234: OK
180/180 1357825755: OK
Saved 180 rows to s3://atlas-acm-research-f26/satellite/metadata/naip_metadata_recovered.csv


In [12]:
print(recovered_metadata['status'].value_counts(dropna=False))
display(recovered_metadata.head())
display(recovered_metadata.loc[recovered_metadata['status'] != 'OK', ['id', 's3_key', 'status', 'notes']])


status
OK    180
Name: count, dtype: int64


,id,s3_key,status,region_geojson,bounds_wgs84,candidate_scene_ids,candidate_scene_dates,scene_count,newest_candidate_date,year,notes,metadata_provenance,source_metadata_key,recovered_at_utc,lat,lon,buffer_meters,image_width_px,image_height_px
0,0007805491,satellite/raw/0007805491.png,OK,"{""geodesic"": false, ""type"": ""Polygon"", ""coordi...","[[[-82.81785870455126, 40.097958491078316], [-...","[""USDA/NAIP/DOQQ/m_4008258_ne_17_030_20230521_...","[""2023-05-21T00:00:00+00:00"", ""2023-05-21T00:0...",2,2023-05-21T00:00:00+00:00,2023,Candidate scenes and crop are reconstructed fr...,reconstructed_not_verified_historical_provenance,dataset/im3_atlas_cleaned.csv,2026-10-28T05:11:25.745089+00:00,40.100657,-82.814358,300,1000,1000
1,0014593270,satellite/raw/0014593270.png,OK,"{""geodesic"": false, ""type"": ""Polygon"", ""coordi...","[[[-81.26510147217753, 35.58607273841938], [-8...","[""USDA/NAIP/DOQQ/m_3508130_ne_17_060_20220927""]","[""2022-09-27T00:00:00+00:00""]",1,2022-09-27T00:00:00+00:00,2022,Candidate scenes and crop are reconstructed fr...,reconstructed_not_verified_historical_provenance,dataset/im3_atlas_cleaned.csv,2026-10-28T05:11:27.307110+00:00,35.588771,-81.261809,300,1000,1000
2,0014930068,satellite/raw/0014930068.png,OK,"{""geodesic"": false, ""type"": ""Polygon"", ""coordi...","[[[-106.54609287400199, 35.04724335838728], [-...","[""USDA/NAIP/DOQQ/m_3510660_se_13_060_20220601""]","[""2022-06-01T00:00:00+00:00""]",1,2022-06-01T00:00:00+00:00,2022,Candidate scenes and crop are reconstructed fr...,reconstructed_not_verified_historical_provenance,dataset/im3_atlas_cleaned.csv,2026-10-28T05:11:28.995293+00:00,35.049942,-106.542822,300,1000,1000
3,0014997588,satellite/raw/0014997588.png,OK,"{""geodesic"": false, ""type"": ""Polygon"", ""coordi...","[[[-77.2398294486869, 37.48006466252088], [-77...","[""USDA/NAIP/DOQQ/m_3707739_nw_18_060_20231013_...","[""2023-10-13T00:00:00+00:00""]",1,2023-10-13T00:00:00+00:00,2023,Candidate scenes and crop are reconstructed fr...,reconstructed_not_verified_historical_provenance,dataset/im3_atlas_cleaned.csv,2026-10-28T05:11:30.503678+00:00,37.482763,-77.236455,300,1000,1000
4,0015884451,satellite/raw/0015884451.png,OK,"{""geodesic"": false, ""type"": ""Polygon"", ""coordi...","[[[-74.28901011304815, 40.64105441101771], [-7...","[""USDA/NAIP/DOQQ/m_4007422_se_18_030_20230809_...","[""2023-08-09T00:00:00+00:00""]",1,2023-08-09T00:00:00+00:00,2023,Candidate scenes and crop are reconstructed fr...,reconstructed_not_verified_historical_provenance,dataset/im3_atlas_cleaned.csv,2026-10-28T05:11:32.558471+00:00,40.643753,-74.285481,300,1000,1000


,id,s3_key,status,notes
